# Semantic Chunking for RAG via BiLSTM


## 0. Setup

In [ ]:
!pip -q install datasets sentence-transformers faiss-cpu mwparserfromhell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = os.environ.get('RAG_PROJECT_DIR')
if not PROJECT_DIR:
    raise RuntimeError('Set RAG_PROJECT_DIR to the project directory before running this notebook.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
# Resolve script paths from the project root.
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print(C.summary())

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())

## Smoke test


In [ ]:
from rag_chunk import smoke
_ = smoke.run_smoke()

## Phase 1: Wikipedia data preparation


In [ ]:
from rag_chunk import wiki_data
stats = wiki_data.prepare_dataset(C.N_WIKI_ARTICLES)
stats

## Phase 2: Offline sentence embedding


In [ ]:
from rag_chunk import embedding
embedding.embed_offline()

## Phase 3: Train the BiLSTM boundary detector


In [ ]:
from rag_chunk import training
train_stats = training.train_model()
train_stats['test_boundary_f1']

## Phase 4: Build RAG indices on Natural Questions


In [ ]:
from rag_chunk import retrieval, training
model = training.load_model()
built = retrieval.build_indexes(model)
print('bilstm chunks:', len(built['bilstm'].chunk_texts),
      '| fixed chunks:', len(built['fixed'].chunk_texts),
      '| questions:', len(built['questions']))

## Phase 5: Evaluate Recall@k and Boundary F1


In [ ]:
from rag_chunk import evaluation, training
model = training.load_model()
results = evaluation.evaluate_all(model)
results

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_FIGURE))

## Phase 6: Chunking sweep optimizer


In [ ]:
from rag_chunk import sweep, training
model = training.load_model()
# quick=True uses the preview grid; quick=False uses the full grid.
rows = sweep.run_sweep(model, quick=True)
print(f"\n{len(rows)} configs swept -> artifacts/results/latest/")

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_LATEST_DIR / C.RECALL_PLOT_PNG))

## Phase 7: Train Transformer boundary model (Stage 2)


In [ ]:
from rag_chunk import training
tf_stats = training.train_model(model_type="transformer")
tf_stats['test_boundary_f1']

## Phase 8: Compare Fixed, BiLSTM, and Transformer (Stage 2)


In [ ]:
from rag_chunk import sweep, training
bilstm = training.load_model("bilstm")
transformer = training.load_model("transformer")
# quick=True uses the preview grid; quick=False uses the full grid.
rows = sweep.run_sweep(bilstm, transformer_model=transformer, quick=True)
print(f"\n{len(rows)} configs swept (fixed + bilstm + transformer) -> artifacts/results/latest/")

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_LATEST_DIR / C.RECALL_PLOT_PNG))

## Stage 3: BGE retrieval embedding ablation

Archive the Stage 2 grid before Stage 3 overwrites `artifacts/results/latest/`.


In [ ]:
# Full Stage 2 grid required for the matched Stage 3 comparison.
!python scripts/8_sweep_with_transformer.py
!python scripts/save_stage_results.py --stage stage2

In [ ]:
# Full Stage 3 BGE retrieval sweep.
!python scripts/9_sweep_bge_retrieval.py


In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.SWEEP_RESULTS_CSV,
    C.BEST_CONFIG_JSON,
    C.FAIR_TABLE_CSV,
    'stage2_vs_stage3_matched.csv',
    C.RECALL_PLOT_PNG,
    C.MODEL_PLOT_PNG,
    'stage3_bge_retrieval_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / 'recall_vs_chunk_size.png')))
display(Image(filename=str(latest / 'model_comparison.png')))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / 'sweep_results.csv'))
display(pd.read_csv(latest / 'fair_comparison_table.csv'))

## Stage 4: Hybrid retrieval ablation


In [ ]:
!python scripts/save_stage_results.py --stage stage3

In [ ]:
!python scripts/10_sweep_hybrid_retrieval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.HYBRID_SWEEP_CSV,
    C.HYBRID_BEST_JSON,
    C.HYBRID_MATCHED_CSV,
    'stage3_vs_stage4_bge_check.csv',
    C.HYBRID_SCATTER_PNG,
    C.HYBRID_RETRIEVER_PLOT_PNG,
    'stage4_hybrid_retrieval_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / C.HYBRID_SCATTER_PNG)))
display(Image(filename=str(latest / C.HYBRID_RETRIEVER_PLOT_PNG)))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.HYBRID_MATCHED_CSV))
display(pd.read_csv(latest / 'stage3_vs_stage4_bge_check.csv'))

In [ ]:
# Requires a successful Stage 3 baseline comparison.
!python scripts/save_stage_results.py --stage stage4

## Stage 5: Cross-encoder reranking


In [ ]:
# Requires stage3/final and an archived results/latest directory.
!python scripts/11_sweep_reranker.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.RERANK_SWEEP_CSV,
    C.RERANK_BEST_JSON,
    C.RERANK_MATCHED_CSV,
    'stage3_vs_stage5_bge_check.csv',
    C.RERANK_SCATTER_PNG,
    C.RERANK_COMPARISON_PNG,
    'stage5_reranker_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / C.RERANK_SCATTER_PNG)))
display(Image(filename=str(latest / C.RERANK_COMPARISON_PNG)))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.RERANK_MATCHED_CSV))
display(pd.read_csv(latest / 'stage3_vs_stage5_bge_check.csv'))

In [ ]:
# Requires a successful Stage 3 baseline comparison.
!python scripts/save_stage_results.py --stage stage5

## Stage 6: Larger-corpus evaluation


In [ ]:
# Requires stage5/final; compares all rows with the archived Stage 5 results.
!python scripts/12_large_eval.py --check

In [ ]:
# Resume: latest/stage6_checkpoint_large.jsonl. --fresh starts a new run.
!python scripts/12_large_eval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.STAGE6_RESULTS_CSV,
    C.STAGE6_MATCHED_CSV,
    C.STAGE6_DIRECTION_CSV,
    C.STAGE6_SUMMARY_MD,
    'stage6_check_vs_stage5.csv',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.STAGE6_MATCHED_CSV))
display(pd.read_csv(latest / C.STAGE6_DIRECTION_CSV))
display(pd.read_csv(latest / 'stage6_check_vs_stage5.csv'))

In [ ]:
!python scripts/13_stage6_plots.py

from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
display(Image(filename=str(latest / C.STAGE6_SIZE_PLOT_PNG)))
display(Image(filename=str(latest / C.STAGE6_DELTA_PLOT_PNG)))

In [ ]:
# Requires completed check-mode, large evaluation, and figures.
!python scripts/save_stage_results.py --stage stage6

## Stage 7: TriviaQA evaluation

Gold documents use distant supervision; see `docs/stage7_cross_dataset.md`.


In [ ]:
# Requires stage3/final; compares all 30 BGE configurations.
!python scripts/15_cross_dataset_eval.py --check

In [ ]:
# Resume: latest/stage7_checkpoint_trivia.jsonl. --fresh starts a new run.
!python scripts/15_cross_dataset_eval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.STAGE7_RESULTS_CSV,
    C.STAGE7_MATCHED_CSV,
    C.STAGE7_DIRECTION_CSV,
    C.STAGE7_SUMMARY_MD,
    C.STAGE7_SCATTER_PNG,
    'stage7_check_vs_stage3.csv',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.STAGE7_MATCHED_CSV))
display(pd.read_csv(latest / C.STAGE7_DIRECTION_CSV))
display(pd.read_csv(latest / 'stage7_check_vs_stage3.csv'))
display(Image(filename=str(latest / C.STAGE7_SCATTER_PNG)))

In [ ]:
# Requires completed check-mode and TriviaQA evaluation.
!python scripts/save_stage_results.py --stage stage7

## Stage 8: Fine-tune the cross-encoder reranker

Run the final evaluation only after a GO decision on the dev bench; see `docs/stage8_reranker_finetune.md`.


In [ ]:
!python scripts/16_build_rerank_train_data.py

In [ ]:
# Resume after epoch 1: --init-model "$RAG_DATA_ROOT/models/bge_reranker_ft/epoch1" --epochs 1
!python scripts/17_train_reranker.py

In [ ]:
# GO permits final evaluation; NO-GO archives the negative result.
# GRAY-ZONE permits at most one retry under the experiment protocol.
!python scripts/18_eval_reranker_ft.py --dev

In [ ]:
# Requires GO and exact reproduction of the archived Stage 6 baseline.
!python scripts/18_eval_reranker_ft.py

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

for name in [C.STAGE8_DEV_RESULTS_CSV, C.STAGE8_RESULTS_CSV,
             C.STAGE8_MATCHED_CSV, C.STAGE8_CHECK_CSV]:
    p = latest / name
    if p.exists():
        print(name)
        display(pd.read_csv(p))
if (latest / C.STAGE8_DELTA_PNG).exists():
    display(Image(filename=str(latest / C.STAGE8_DELTA_PNG)))

In [ ]:
# Archive dev results after NO-GO, or final results after GO and baseline checks.
!python scripts/save_stage_results.py --stage stage8

## Route D: Start the Gradio demo

Creates a public sharing link.


In [ ]:
%pip install -q gradio
!python scripts/19_demo.py --share